# 04 — Definición de KPIs

Fase 4 del proyecto. Aquí definimos y calculamos los indicadores clave, los justificamos desde el negocio, los relacionamos con las hipótesis e interpretamos cada uno. Los valores sirven de referencia para verificar las medidas del dashboard.

## 1. Definición de KPIs

Definimos seis indicadores para apoyar las decisiones de una productora con capital limitado: cuánto invertir, cuándo estrenar y qué tipo de película hacer. Cubrimos el retorno habitual, la proporción que alcanza el umbral acordado, la exposición a no recuperar el presupuesto, el capital requerido y el volumen de taquilla. Para comparar temporadas conservamos también la medida utilizada en la prueba estadística.

Cada ficha establece nombre, fórmula, población, unidad, justificación de negocio, relación con las hipótesis e interpretación. Aquí dejamos el contrato de cálculo; los valores generales, las comparaciones por segmento y la tabla de referencia para Power BI se desarrollarán en la siguiente sección de este notebook.

Usamos las definiciones del notebook 02 y las pruebas del notebook 03. Cuando mencionamos H1 hablamos de presupuesto y retorno; H2 corresponde a temporada de estreno y H3 a género y rentabilidad. Un KPI resume el desempeño observado: su valor, por sí solo, no demuestra una causa ni sustituye la prueba estadística.

### 1.1 Entrada y comprobación de las variables disponibles

Leemos `data/processed/peliculas_variables.csv`, el resultado de preparación y selección de variables. No dependemos de ejecutar los notebooks anteriores en el mismo kernel. Las rutas funcionan desde `notebooks/` o desde la raíz del repositorio.

La lectura explícita de lógicos con faltantes conserva la diferencia entre `rentable = False` y una película cuya rentabilidad no se puede evaluar. No cambiamos la limpieza ni recalculamos los filtros acordados.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

kpi_ruta = next((carpeta / "data/processed/peliculas_variables.csv"
                 for carpeta in (Path.cwd(), Path.cwd().parent)
                 if (carpeta / "data/processed/peliculas_variables.csv").is_file()), None)
if kpi_ruta is None:
    raise FileNotFoundError(
        "Necesitamos data/processed/peliculas_variables.csv. "
        "Ejecutamos desde notebooks/ o desde la raíz del proyecto."
    )

kpi_peliculas = pd.read_csv(
    kpi_ruta, parse_dates=["release_date"],
    dtype={"anio": "Int64", "mes": "Int64", "decada": "Int64",
           "temporada_alta": "boolean", "rentable": "boolean",
           "estrenada": "boolean", "monto_dudoso": "boolean", "base_financiera": "boolean"},
)
kpi_columnas_requeridas = {
    "id", "release_date", "anio", "mes", "decada", "temporada_alta", "original_language",
    "genero_principal", "genero_agrupado", "budget", "revenue", "roi", "multiplicador",
    "rentable", "estrenada", "monto_dudoso", "base_financiera",
}
assert kpi_columnas_requeridas.issubset(kpi_peliculas.columns), "Faltan variables para los indicadores"
assert kpi_peliculas["id"].notna().all() and kpi_peliculas["id"].is_unique
assert kpi_peliculas[["estrenada", "monto_dudoso", "base_financiera"]].notna().all().all()
kpi_base = kpi_peliculas.loc[kpi_peliculas["base_financiera"]].copy()
assert not kpi_base.empty, "No hay películas elegibles para verificar las definiciones financieras"

display(pd.Series({
    "Películas del archivo": len(kpi_peliculas),
    "Población financiera disponible": len(kpi_base),
    "De la población financiera, desde 2000": int(kpi_base["anio"].ge(2000).sum()),
    "De la población financiera, sin género agrupado": int(kpi_base["genero_agrupado"].isna().sum()),
}, name="películas").to_frame())

,películas
Películas del archivo,4803
Población financiera disponible,3204
"De la población financiera, desde 2000",2265
"De la población financiera, sin género agrupado",1


### 1.2 Población y reglas comunes

Llamamos **F** al conjunto de películas con `base_financiera = True`: estrenadas, con presupuesto y recaudación reportados y sin montos menores a USD 10.000, según la preparación existente. Para un filtro o segmento **S** usamos **Fₛ = F ∩ S** y **nₛ** es su número de películas únicas. Los seis indicadores comparten ese conjunto, incluso presupuesto y recaudación, para comparar retorno y volumen sobre las mismas películas.

En el archivo actual hay 3.204 películas elegibles. Esa cantidad es una comprobación de entrada, no un denominador fijo: al filtrar por género, década, idioma o temporada, el denominador debe recalcularse dentro de Fₛ.

- **Una película, una observación.** Contamos `id` únicos. Al usar el modelo dimensional, el equivalente es `fact_peliculas.pelicula_id`; los puentes solo seleccionan películas, no replican sus importes.
- **Sin población válida, sin valor.** Si nₛ = 0, los seis indicadores se presentan como no disponibles (`NaN` en Python o blanco en Power BI). Una taquilla de cero o un 0 % comunicarían un resultado que no observamos. En la suma usamos `sum(min_count=1)` después de seleccionar la población.
- **Retorno registrado como razón.** `roi = multiplicador - 1` se conserva en escala decimal. Para mostrarlo en Python como porcentaje multiplicamos por 100 una sola vez. En Power BI puede conservarse la fracción y aplicarse formato de porcentaje; no se combina ese formato con otra multiplicación por 100. Lo mismo vale para las dos proporciones.
- **Cortes consistentes.** `multiplicador < 1` significa que la taquilla no recuperó el presupuesto; `multiplicador >= 2` es el criterio aproximado de rentable. Los casos con multiplicador igual a 1 no entran al primer grupo, y los iguales a 2 sí entran al segundo. Entre 1 y menos de 2 existe un grupo intermedio.
- **Montos nominales.** Presupuesto y recaudación se expresan en USD de la época, sin ajuste por inflación. Estos indicadores no miden utilidad neta, porque faltan publicidad, participación de salas y otros costos e ingresos.

Antes de continuar verificamos que las variables disponibles cumplan esas definiciones. Son controles sobre las columnas existentes; todavía no calculamos los valores de los seis KPIs.

In [2]:
kpi_umbral_rentable = 2
kpi_monto_minimo = 10_000
kpi_meses_temporada_alta = [5, 6, 7, 8, 11, 12]

assert kpi_base[["budget", "revenue", "roi", "multiplicador", "rentable"]].notna().all().all()
assert np.isfinite(kpi_base[["budget", "revenue", "roi", "multiplicador"]]).all().all()
assert kpi_base[["budget", "revenue"]].ge(kpi_monto_minimo).all().all()
assert kpi_base["estrenada"].all() and not kpi_base["monto_dudoso"].any()
assert kpi_base["multiplicador"].gt(0).all(), "El logaritmo necesita multiplicadores positivos"
assert np.allclose(kpi_base["multiplicador"], kpi_base["revenue"] / kpi_base["budget"])
assert np.allclose(kpi_base["roi"], kpi_base["multiplicador"] - 1)
assert kpi_base["rentable"].eq(kpi_base["multiplicador"].ge(kpi_umbral_rentable)).all()

kpi_con_fecha = kpi_base["release_date"].notna()
assert kpi_base["temporada_alta"].isna().eq(kpi_base["release_date"].isna()).all()
assert kpi_base.loc[kpi_con_fecha, "temporada_alta"].eq(
    kpi_base.loc[kpi_con_fecha, "mes"].isin(kpi_meses_temporada_alta)).all()
print("Entrada verificada: identidad, población financiera, razones y umbrales consistentes.")

Entrada verificada: identidad, población financiera, razones y umbrales consistentes.


### 1.3 Catálogo de los seis indicadores

Registramos identificadores estables para referirnos a los mismos KPIs en el cálculo y en el dashboard. La columna `expresion_python` es una referencia de implementación sobre una tabla `base` que ya representa Fₛ; se guarda como texto y no se evalúa en esta sección. Antes de usar cualquier expresión se comprueba que el grupo no esté vacío.

Las fórmulas, la interpretación y las limitaciones se detallan en las fichas siguientes. La vinculación con una hipótesis indica qué pregunta ayuda a seguir el indicador; no implica que todas las diferencias de ese KPI hayan sido sometidas a una prueba específica.

In [3]:
kpi_definiciones = [
    {"id": "KPI01", "nombre": "ROI mediano", "unidad": "%",
     "hipotesis": "H1", "columnas": ["roi"],
     "expresion_python": '100 * base["roi"].median()'},
    {"id": "KPI02", "nombre": "Películas rentables según umbral 2x", "unidad": "%",
     "hipotesis": "H3", "columnas": ["rentable", "multiplicador"],
     "expresion_python": '100 * base["rentable"].mean()'},
    {"id": "KPI03", "nombre": "Películas que no recuperan el presupuesto", "unidad": "%",
     "hipotesis": "H1, H2", "columnas": ["multiplicador"],
     "expresion_python": '100 * base["multiplicador"].lt(1).mean()'},
    {"id": "KPI04", "nombre": "Presupuesto mediano", "unidad": "USD",
     "hipotesis": "H1", "columnas": ["budget"],
     "expresion_python": 'base["budget"].median()'},
    {"id": "KPI05", "nombre": "Recaudación total", "unidad": "USD",
     "hipotesis": "H1", "columnas": ["revenue"],
     "expresion_python": 'base["revenue"].sum(min_count=1)'},
    {"id": "KPI06", "nombre": "Multiplicador típico (media geométrica)", "unidad": "veces (x)",
     "hipotesis": "H2", "columnas": ["multiplicador"],
     "expresion_python": '10 ** np.log10(base["multiplicador"]).mean()'},
]
kpi_catalogo = pd.DataFrame(kpi_definiciones).set_index("id")
with pd.option_context("display.max_colwidth", 55):
    display(kpi_catalogo[["nombre", "unidad", "hipotesis"]])

,nombre,unidad,hipotesis
id,,,
KPI01,ROI mediano,%,H1
KPI02,Películas rentables según umbral 2x,%,H3
KPI03,Películas que no recuperan el presupuesto,%,"H1, H2"
KPI04,Presupuesto mediano,USD,H1
KPI05,Recaudación total,USD,H1
KPI06,Multiplicador típico (media geométrica),veces (x),H2


### 1.4 KPI01 — ROI mediano

**Fórmula.** Para cada película i de Fₛ definimos Rᵢ = revenueᵢ / budgetᵢ. El indicador es:

$$\text{ROI mediano}_s = 100 \times \operatorname{mediana}_{i \in F_s}(R_i - 1)$$

**Población y unidad.** Películas únicas de Fₛ; porcentaje. Usamos el `roi` ya calculado, que está en escala decimal.

**Justificación de negocio.** Describe el retorno central de las películas comparables sin dejar que unas pocas producciones con multiplicadores enormes dominen un promedio. Ayuda a contrastar cuánto rinde cada dólar con cuánto capital exige un segmento.

**Relación con H1.** La hipótesis 1 estudia presupuesto frente a multiplicador; restar uno no cambia sus rangos. Este KPI permite seguir el retorno por nivel de presupuesto y época, pero no sustituye el coeficiente ni el valor p. El resultado de H1 cambió al controlar por época, por lo que debemos mostrar también la vista desde 2000.

**Cómo lo interpretamos.** Un ROI individual de 0 % significa recaudar lo mismo que el presupuesto; 100 % equivale a un multiplicador de 2x. El KPI resume la mediana de esos retornos, no el porcentaje de películas rentables ni el retorno conjunto de una cartera. Un valor más alto indica mayor retorno central observado, sujeto a la composición del grupo y al criterio de datos válidos.

**Límite.** No lo calculamos como `(sum(revenue) - sum(budget)) / sum(budget)`: esa sería otra medida, ponderada por el tamaño de las inversiones. Tampoco es utilidad neta para la productora.

### 1.5 KPI02 — Porcentaje de películas rentables según el umbral de 2x

**Fórmula.** Usamos la condición `rentable` acordada en la preparación:

$$\text{Rentables}_s = 100 \times \frac{\#\{i \in F_s : R_i \geq 2\}}{n_s}$$

**Población y unidad.** Todas las películas únicas de Fₛ; porcentaje. El denominador incluye también las que no alcanzaron 2x, pero no las películas fuera de la base financiera, cuyo `rentable` está vacío.

**Justificación de negocio.** Mide con qué frecuencia histórica un grupo alcanzó la regla de retorno definida por el equipo. Ayuda a comparar opciones de contenido y a distinguir un segmento con éxitos excepcionales de otro que alcanza el umbral con mayor frecuencia.

**Relación con H3.** Es la proporción que compara la prueba de género y rentabilidad. Para reproducir esa comparación usamos `genero_agrupado`, con los mismos grupos de la hipótesis 3. La asociación observada no convierte el porcentaje histórico en una probabilidad garantizada para la próxima película.

**Cómo lo interpretamos.** Un porcentaje mayor significa que más películas del grupo recaudaron al menos el doble de su presupuesto. Las diferencias entre porcentajes se comunican en **puntos porcentuales**; no confundimos esa diferencia con un cambio porcentual relativo.

**Límite.** El umbral 2x es una aproximación del proyecto, no una medición exacta del punto de equilibrio. No inventamos una meta de 50 %, 60 % u otro valor que la productora no haya definido. En grupos sin observaciones el resultado queda no disponible.

### 1.6 KPI03 — Porcentaje de películas que no recuperan el presupuesto en taquilla

**Fórmula.** Contamos las películas cuya recaudación es menor que su presupuesto:

$$\text{No recuperan presupuesto}_s = 100 \times \frac{\#\{i \in F_s : R_i < 1\}}{n_s}$$

**Población y unidad.** Películas únicas de Fₛ; porcentaje. Igual que en KPI02, el denominador se recalcula en cada grupo.

**Justificación de negocio.** Una productora con capital limitado necesita vigilar el riesgo de que la taquilla ni siquiera cubra el presupuesto reportado, además del retorno central. Un ROI mediano favorable puede coexistir con una proporción relevante de resultados por debajo de 1x.

**Relación con H1 y H2.** Las secciones de magnitud del notebook 03 comparan este riesgo por presupuesto y temporada. Esas proporciones se estudiaron de manera descriptiva: los valores p de Spearman y Welch no constituyen pruebas específicas de diferencias de este porcentaje.

**Cómo lo interpretamos.** Un valor menor indica menos películas que quedaron por debajo del presupuesto en la taquilla registrada. No lo llamamos porcentaje de quiebras ni de pérdidas contables: no conocemos el conjunto de costos e ingresos del productor.

**Límite.** No es `100 - KPI02`. Las películas con 1x ≤ Rᵢ < 2x no pertenecen a ninguno de esos dos numeradores. Con Rᵢ = 1 no marcamos falta de recuperación del presupuesto; con Rᵢ = 2 sí marcamos rentable según la regla del proyecto.

### 1.7 KPI04 — Presupuesto mediano

**Fórmula.**

$$\text{Presupuesto mediano}_s = \operatorname{mediana}_{i \in F_s}(budget_i)$$

**Población y unidad.** Películas únicas de Fₛ; USD nominales. Si lo presentamos en millones, dividimos el resultado entre un millón y dejamos visible la unidad.

**Justificación de negocio.** Describe el capital de producción habitual de las películas que estamos comparando y ayuda a evaluar si una estrategia es compatible con una productora de recursos limitados. Elegimos la mediana por la asimetría de los presupuestos observada en el diagnóstico y las hipótesis.

**Relación con H1.** Es el contexto de inversión para el retorno. La hipótesis 1 distingue gastar más de rendir más por dólar, y su control por época evita atribuir al presupuesto una relación que puede venir de mezclar periodos.

**Cómo lo interpretamos.** Un valor alto describe producciones con una escala de inversión mayor; uno bajo describe menor requerimiento central de capital. Ninguna dirección es mejor por sí sola: lo leemos junto con ROI mediano, proporción rentable y riesgo. No conocemos el capital de la productora para fijar una meta numérica de presupuesto.

**Límite.** No representa el presupuesto total necesario para financiar todas las películas de un grupo. Al comparar décadas recordamos que no ajustamos los USD por inflación.

### 1.8 KPI05 — Recaudación total

**Fórmula.**

$$\text{Recaudación total}_s = \sum_{i \in F_s} revenue_i$$

**Población y unidad.** Películas únicas de Fₛ; USD nominales. Conservamos exactamente la población financiera usada por los otros cinco indicadores. No mezclamos una taquilla calculada con todas las películas que reportaron `revenue` con un retorno calculado sobre una población menor.

**Justificación de negocio.** Muestra el volumen de taquilla observado en el segmento y aporta contexto sobre la escala comercial. Debe acompañarse de nₛ, porque un grupo con más películas puede recaudar más solo por su tamaño.

**Relación con H1.** Ayuda a distinguir el dinero recaudado del rendimiento por cada dólar invertido, que es la pregunta de la hipótesis. Una recaudación mayor no demuestra un ROI mayor ni confirma una relación causal con el presupuesto.

**Cómo lo interpretamos.** Un total mayor indica más taquilla agregada dentro del filtro y de la población declarada. Se puede expresar en millones, con unidad visible. No lo interpretamos como ganancia, ingreso neto de la productora ni tamaño de todo el mercado cinematográfico.

**Límite.** El total general se obtiene de las películas únicas. No sumamos subtotales de géneros multivaluados o directores para reconstruirlo, porque una película puede aparecer en varios segmentos. En grupos vacíos la suma debe quedar no disponible, no en cero.

### 1.9 KPI06 — Multiplicador típico: media geométrica

**Fórmula.** Devolvemos a la escala original el promedio del logaritmo del multiplicador:

$$G_s = 10^{\frac{1}{n_s}\sum_{i \in F_s}\log_{10}(R_i)}$$

**Población y unidad.** Películas únicas de Fₛ, con multiplicadores positivos; veces (x). La población financiera ya cumple esa condición. La media geométrica fue introducida en la sección 2.2 del notebook 03 como lectura de negocio de la prueba de temporada; no es un concepto desarrollado en las guías de clase.

**Justificación de negocio.** Permite seguir la misma medida de retorno que se comparó estadísticamente entre temporada alta y resto del año. Así evitamos presentar una mediana o un promedio aritmético como si fueran el resultado de la prueba sobre promedios de logaritmos.

**Relación con H2.** Si calculamos G_alta y G_resto sobre la misma población y periodo, su cociente es `10 ** (media_log_alta - media_log_resto)`, la escala de comparación usada por Welch. Se mantienen los meses de temporada alta y el control desde 2000 del notebook 03.

**Cómo lo interpretamos.** Describe el centro multiplicativo del retorno observado. Un Gₛ mayor representa mayor retorno en esa escala, pero no indica la fracción de rentables. Un cociente G_alta / G_resto mayor que 1 señala ventaja descriptiva de temporada alta, y un cociente igual a 1 indica igualdad de esta medida; la incertidumbre se lee con la prueba y su intervalo correspondientes.

**Límite.** No es la mediana ni `sum(revenue) / sum(budget)`. Para un total o un nuevo segmento se recalcula con las películas originales; no promediamos las medias geométricas de grupos ni extrapolamos el valor p de toda la base a cada filtro del dashboard.

### 1.10 Reglas de segmentación, comparación y presentación

| Vista | Variable preparada | Regla de lectura |
|---|---|---|
| Género y comparación con H3 | `genero_agrupado` | Mantener los grupos de la prueba; informar las películas sin género. |
| Género principal descriptivo | `genero_principal` | Una categoría por película; declarar que se toma la primera de la fuente. |
| Todos los géneros | Puente `pelicula_genero` | Seleccionar identificadores únicos antes de calcular; los grupos se solapan. |
| Década y control de época | `decada`, `anio` | Mostrar toda la base y la vista desde 2000 sin reemplazar una por otra; la década de 2010 está incompleta. |
| Idioma original | `original_language` | Informar nₛ; los grupos pequeños y el predominio del inglés limitan la comparación. |
| Temporada de estreno | `temporada_alta` | Mayo–agosto y noviembre–diciembre; comparar grupos en el mismo periodo. |
| Talento en el modelo | Puente `pelicula_director` | Seleccionar películas únicas aunque tengan varios directores. |

La base financiera contiene una película sin género agrupado. Se incluye en los indicadores generales, pero la prueba de H3 la excluye. Para reproducir esa prueba se declara esa exclusión; si una vista descriptiva quiere mantener cobertura completa, puede mostrarla como **sin género registrado** sin modificar el archivo preparado ni añadirla a la prueba original. La misma regla de declarar faltantes se aplica a cualquier dimensión de segmentación.

Cada resultado debe mostrar el periodo o filtro, nₛ y la unidad. Las medianas y proporciones se recalculan desde las películas al cambiar el filtro: no promediamos los porcentajes o las medianas de los segmentos para obtener el total. Recaudación sí se suma sobre conjuntos sin solapamiento. Las diferencias entre porcentajes se expresan en puntos porcentuales y los cocientes de multiplicadores en veces.

Los umbrales 1x y 2x tienen significados definidos, pero no contamos con metas de la productora para asignar semáforos de desempeño. Las comparaciones históricas no eliminan el sesgo de selección de películas con cifras, ni convierten asociación en causalidad. Un segmento con pocos casos debe informar su soporte y no se presenta como evidencia concluyente.

### 1.11 Comprobación del catálogo y continuidad del cálculo

Dejamos seis fichas con fórmulas, población y forma de interpretación; el catálogo relaciona cada una con las variables y las hipótesis ya implementadas. La siguiente sección podrá usar `kpi_base`, seleccionar Fₛ y aplicar las expresiones definidas, comprobando primero si el grupo está vacío.

El cálculo posterior deberá presentar los valores generales y por género, década e idioma, y conservar la comparación por temporada y época cuando corresponda. Su tabla de referencia indicará los filtros y nₛ para contrastar después las medidas de Power BI. Aquí verificamos la disponibilidad y coherencia de las entradas y la integridad del catálogo, sin exportar ni modificar los datos preparados.

In [4]:
assert len(kpi_catalogo) >= 5
assert kpi_catalogo.index.is_unique and kpi_catalogo["nombre"].is_unique
assert kpi_catalogo[["nombre", "unidad", "hipotesis", "expresion_python"]].notna().all().all()
assert kpi_catalogo["unidad"].isin(["%", "USD", "veces (x)"]).all()
kpi_hipotesis_cubiertas = set()
for kpi_definicion in kpi_definiciones:
    assert set(kpi_definicion["columnas"]).issubset(kpi_base.columns)
    kpi_hipotesis_cubiertas.update(kpi_definicion["hipotesis"].split(", "))
assert kpi_hipotesis_cubiertas == {"H1", "H2", "H3"}
print(f"Definición terminada: {len(kpi_catalogo)} KPIs con variables verificadas "
      "y relación con las tres hipótesis. Fórmulas listas para el cálculo posterior.")

Definición terminada: 6 KPIs con variables verificadas y relación con las tres hipótesis. Fórmulas listas para el cálculo posterior.
